## completeness check

This notebook introduces `asserted_inference`; after running it you can record a functional-completeness judgment that ties child claims (the action-level evidence) to a parent claim (the functional architecture is complete), following Hawkins §3.1.

The Chapter 4 model now has a functional decomposition: `ApplyHeat` sequences power input through a calculation to an energy output, with item types naming the flows. The question is whether that decomposition is complete — does every input flow contribute to an output? An `asserted_inference` (Hawkins §3.1) records this judgment: a parent claim supported by child claims, rather than directly by evidence. This is the first use of the inference record type.

In [ ]:
from pathlib import Path
import opensysml
from toaster.report import format_diagnostics

conn = opensysml.connect(version="v0.9.0")
source = Path("../../models/ch04-cumulative.sysml").read_text()
print(source)
model = conn.load_from_content(source, strict=False)
assert model.ok, f"Model failed: {format_diagnostics(model.diagnostics)}"

The `ch04-cumulative.sysml` file adds `action def ApplyHeat` with sequential steps (`first start; then calculate; then done`) and a nested `calculate` action that calls `DeliveredEnergy`. Three `item def` types — `Start`, `Finish`, `Cancel` — represent items flowing between action steps. This is the functional decomposition of the heating operation: inputs, process, and outputs all named.

In [ ]:
# Negative control: an action def that assigns to an out parameter via
# an undefined action definition raises "unresolved reference".
bad_source = """
package Bad {
    private import ScalarValues::*;
    action def BadDecomp {
        in x : Real;
        out y : Real;
        first start;
        then action step : UndefinedActionDef;
        then done;
    }
}
"""
bad = conn.load_from_content(bad_source, strict=False)
assert not bad.ok
print("Expected error:", bad.diagnostics[0].message)

In [ ]:
inference_record = ReviewRecord(
    identifier="AI-C04",
    kind="asserted_inference",
    claim="The ApplyHeat action decomposition is functionally complete: all inputs are consumed and the output is assigned.",
    model_ref="ToasterDemo::ApplyHeat",
    content_hash=hash_content(source),
    scope="ToasterDemo",
    criteria="Every in parameter feeds at least one sub-action; the out parameter is assigned before done.",
    premises=["AS-C03"],
    assumption_refs=[],
    evidence_refs=["action calculate { assign energy := DeliveredEnergy(power, duration, efficiency); }"],
    rationale="The calculate action consumes all three in parameters (power, duration, efficiency) and assigns the out parameter (energy). No input is unrouted.",
    counterevidence="The model does not capture heat loss or warm-up transients — those flows are absent from this decomposition.",
    residual_uncertainties="Temporal ordering via first/then is syntactic; actual execution semantics are not checked by this model alone.",
    disposition="pending",
    dependency_freshness="current",
    engineering_conclusion="undetermined",
    record_kind="worked_example",
)

errors = validate_record(inference_record)
print(f"Validation errors: {errors}")
print(f"Record kind:       {inference_record.kind}")
conn.close()

The Hawkins §3.1 schema specifies what an `asserted_inference` record must contain, including a non-empty `premises` list (A-F); filling and validating the `ReviewRecord` in Python enacts that schema (O-S); `validate_record()` returning `[]` confirms all required fields are present (E).

Try the chapter exercise in `exercises/ch04/exercise.ipynb`: write an `asserted_inference` record claiming that the `BrewUnit` action decomposition accounts for all inputs and outputs.